In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ---------- Load data ----------
DATA_PATH = Path("..") / "Dataset .csv"
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Dataset not found at {DATA_PATH.resolve()}")
if DATA_PATH.stat().st_size == 0:
    raise ValueError("Dataset file is empty. Copy the real file again.")

df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()

REQUIRED = [
    "Restaurant Name", "City", "Cuisines", "Price range",
    "Aggregate rating", "Votes",
]
missing_cols = [c for c in REQUIRED if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing expected columns: {missing_cols}")

# ---------- Clean ----------
df = df.drop_duplicates().dropna(subset=["Cuisines", "Restaurant Name", "City"]).copy()
df["Price range"] = pd.to_numeric(df["Price range"], errors="coerce")
df["Aggregate rating"] = pd.to_numeric(df["Aggregate rating"], errors="coerce")
df = df.dropna(subset=["Price range", "Aggregate rating"])
df = df[df["Price range"].between(1, 4)]
df = df[df["Aggregate rating"] > 0]  # unrated restaurants are not useful to recommend
df["Price range"] = df["Price range"].astype(int)
df = df.reset_index(drop=True)
print("Restaurants available:", df.shape[0])

# ---------- Build features ----------
# 1) Cuisines -> multi-hot (a restaurant can have several cuisines)
def split_cuisines(text):
    return [c.strip().lower() for c in text.split(",") if c.strip()]

cv = CountVectorizer(tokenizer=split_cuisines, lowercase=False,
                     binary=True, token_pattern=None)
cuisine_matrix = cv.fit_transform(df["Cuisines"])

# 2) Price range -> one-hot (4 columns)
price_matrix = csr_matrix(np.eye(4)[df["Price range"].values - 1])

# 3) Rating -> scaled 0..1
rating_matrix = csr_matrix((df["Aggregate rating"].values / 5.0).reshape(-1, 1))

# Importance of each part in the similarity score
W_CUISINE, W_PRICE, W_RATING = 1.0, 0.8, 0.3

features = hstack([
    cuisine_matrix * W_CUISINE,
    price_matrix * W_PRICE,
    rating_matrix * W_RATING,
]).tocsr()
print("Feature matrix:", features.shape)

# ---------- Recommendation by user preferences ----------
OUT_COLS = ["Restaurant Name", "City", "Cuisines", "Price range",
            "Aggregate rating", "Votes", "Similarity"]

def recommend(cuisines, price_range, city=None, min_rating=0.0, top_n=5):
    wanted = [c.strip().lower() for c in cuisines]
    unknown = [c for c in wanted if c not in cv.vocabulary_]
    if unknown:
        raise ValueError(f"Unknown cuisine(s): {unknown}")
    if price_range not in (1, 2, 3, 4):
        raise ValueError("price_range must be 1, 2, 3 or 4")

    user_cuisine = cv.transform([", ".join(wanted)])
    user_price = csr_matrix(np.eye(4)[[price_range - 1]])
    user_rating = csr_matrix([[1.0]])  # ideal: a highly rated restaurant
    user_vec = hstack([
        user_cuisine * W_CUISINE, user_price * W_PRICE, user_rating * W_RATING
    ]).tocsr()

    scores = cosine_similarity(user_vec, features).ravel()
    mask = (df["Aggregate rating"] >= min_rating).values
    if city:
        if city.strip().lower() not in set(df["City"].str.lower()):
            raise ValueError(f"Unknown city: {city}")
        mask &= (df["City"].str.lower() == city.strip().lower()).values

    out = df[mask].copy()
    out["Similarity"] = scores[mask].round(3)
    out = out.sort_values(["Similarity", "Aggregate rating", "Votes"], ascending=False)
    return out[OUT_COLS].head(top_n).reset_index(drop=True)

# ---------- Recommendation similar to a restaurant ----------
def recommend_similar(name, top_n=5):
    matches = df.index[df["Restaurant Name"].str.lower() == name.strip().lower()]
    if len(matches) == 0:
        raise ValueError(f"Restaurant not found: {name}")
    idx = matches[0]
    scores = cosine_similarity(features[idx], features).ravel()
    out = df.copy()
    out["Similarity"] = scores.round(3)
    out = out[out["Restaurant Name"].str.lower() != name.strip().lower()]
    out = out.sort_values(["Similarity", "Aggregate rating", "Votes"], ascending=False)
    return out[OUT_COLS].head(top_n).reset_index(drop=True)

# ---------- Test with sample user preferences ----------
test_cases = [
    (["north indian"], 2, "New Delhi", 0.0),
    (["chinese"], 2, "New Delhi", 0.0),
    (["italian"], 3, "New Delhi", 0.0),
    (["cafe"], 2, None, 4.0),
    (["north indian", "mughlai"], 3, "New Delhi", 0.0),
]

all_recs = []
for cuisines, price, city, min_rating in test_cases:
    print(f"\n=== {cuisines} | price {price} | city {city} | min rating {min_rating} ===")
    recs = recommend(cuisines, price, city, min_rating)
    print(recs.to_string(index=False) if not recs.empty else "No matches found")
    all_recs.append(recs.assign(Request=str(cuisines) + f" price {price}"))

# ---------- Evaluation: precision@5 ----------
# A recommendation is a "hit" if it serves ALL requested cuisines
# AND matches the requested price range.
def precision_at_k(cuisines, price_range, city=None, min_rating=0.0, k=5):
    recs = recommend(cuisines, price_range, city, min_rating, top_n=k)
    if recs.empty:
        return None
    wanted = {c.strip().lower() for c in cuisines}
    hits = recs.apply(
        lambda r: wanted.issubset(set(split_cuisines(r["Cuisines"])))
        and r["Price range"] == price_range,
        axis=1,
    )
    return float(hits.mean())

print("\nPrecision@5 per test case:")
scores_list = []
for cuisines, price, city, min_rating in test_cases:
    p = precision_at_k(cuisines, price, city, min_rating)
    scores_list.append(p)
    print(cuisines, price, city, "->", p)
valid = [s for s in scores_list if s is not None]
print("Average precision@5:", round(float(np.mean(valid)), 3))

# ---------- Similar-restaurant demo ----------
demo_name = df.sort_values("Votes", ascending=False)["Restaurant Name"].iloc[0]
print(f"\nRestaurants similar to '{demo_name}':")
print(recommend_similar(demo_name).to_string(index=False))

# ---------- Save ----------
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
pd.concat(all_recs).to_csv(OUTPUT_DIR / "sample_recommendations.csv", index=False)
print("\nSaved to", OUTPUT_DIR.resolve())

Restaurants available: 7394
Feature matrix: (7394, 148)

=== ['north indian'] | price 2 | city New Delhi | min rating 0.0 ===
 Restaurant Name      City     Cuisines  Price range  Aggregate rating  Votes  Similarity
    Satvik Rasoi New Delhi North Indian            2               3.8     31       0.999
         Kamal's New Delhi North Indian            2               3.7    286       0.998
      Prem Dhaba New Delhi North Indian            2               3.7    177       0.998
          Chowki New Delhi North Indian            2               3.7     89       0.998
Anand Restaurant New Delhi North Indian            2               3.6    485       0.998

=== ['chinese'] | price 2 | city New Delhi | min rating 0.0 ===
 Restaurant Name      City Cuisines  Price range  Aggregate rating  Votes  Similarity
     Happy Hakka New Delhi  Chinese            2               4.1    173       0.999
       Ding Ding New Delhi  Chinese            2               3.9     93       0.999
       Casa